In [2]:
%config InlineBackend.rc = {"figure.dpi": 72, 'figure.figsize': (6.0, 4.0)}
%matplotlib ipympl

import ase 
import matplotlib.pyplot as plt
import numpy as np
from IPython.display import HTML

import abtem

abtem.config.set({"diagnostics.progress_bar": False})
abtem.config.set({"device": "gpu"});

## STEM 

use model MoS2 to create potential ,probe

In [5]:
atoms = ase.build.mx2(vacuum=3)
atoms = abtem.orthogonalize_cell(atoms) * (3, 2, 1)

probe = abtem.Probe(
    semiangle_cutoff=np.linspace(20, 30, 3),
    defocus=np.linspace(0, 50, 3),
    Cs=np.linspace(0, 5e5, 3),
    astigmatism_angle=np.pi / 4,
    sampling=0.05,
    energy=80e3,
)

potential = abtem.Potential(atoms, projection="infinite", slice_thickness=2)

scan = abtem.GridScan(
    start=(0, 0),
    end=(1 / 3, 1 / 2),
    fractional=True,
    potential=potential,
)

doses = np.geomspace(1e1, 1e7, 7)

measurements = (
    probe.scan(potential, scan=scan)
    .to_image_ensemble()
    .gaussian_filter(0.3)
    .interpolate(0.1)
    .tile((3, 2))
    .compute()
    .poisson_noise(dose_per_area=doses)
    .compute()
)

show method now provide a widget where each ensemble can be separately controlled.

In [6]:
measurements.show(interact=True, cbar=True);

ImageGUI(children=(VBox(children=(SelectionSlider(continuous_update=False, description='Dose [e]', options=(np…